In [1]:
!pip install langchain==0.1.17
!pip install langchain-openai==0.0.6
!pip install langchain-core==0.1.53 
!pip install langchain-community==0.0.33

  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
Using cached langchain_community-0.0.38-py3-none-any.whl (2.0 MB)
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.33
    Uninstalling langchain-community-0.0.33:
      Successfully uninstalled langchain-community-0.0.33


  Using cached langchain_community-0.0.33-py3-none-any.whl.metadata (8.5 kB)
Using cached langchain_community-0.0.33-py3-none-any.whl (1.9 MB)
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.38
    Uninstalling langchain-community-0.0.38:
      Successfully uninstalled langchain-community-0.0.38


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.17 requires langchain-community<0.1,>=0.0.36, but you have langchain-community 0.0.33 which is incompatible.


In [2]:
from openai import OpenAI
client = OpenAI()

#Zero Prompt:	
Model performs tasks without any examples, relying on its pre-existing knowledge.
No examples are provided; the model uses its prior training.
Fast and efficient for general tasks but can be less precise for specific tasks.
Models handle tasks directly without the need for task-specific examples.


In [3]:
response_zero = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Classify this text as Positive or Negative: 'I love this product!'"}]
)
print("Zero-shot result:", response_zero.choices[0].message.content)

Zero-shot result: Positive


#Few Prompt
Model learns from a few examples provided in the prompt to perform the task.
A few examples are given within the prompt to guide the model.
Effective for tasks where a few examples are enough to guide the model.
Models adapt to the task through provided examples in the prompt.

In [4]:
examples = """
Text: 'I love this product!' -> Neutral
Text: 'This is terrible.' -> Neutral
Text: 'Not bad, could be better.' -> Neutral
"""

response_few  = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": examples + "\nText: 'The service was Excellent!' ->"}]
)
print("Few-shot result:", response_few.choices[0].message.content)

Few-shot result: Positive


In [5]:
response_zero_translate = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role":"user","content":"Translate to French: Hello"}])
print("Zero-shot result:", response_zero_translate.choices[0].message.content)

Zero-shot result: Bonjour


In [6]:
response_few_translate = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role":"user","content":"English: Hello\nFrench: Bonjour\nEnglish: Good morning\nFrench: Bonjour\nEnglish: How are you?\nFrench:"}])
print("few-shot result:", response_few_translate.choices[0].message.content)

few-shot result: Comment ça va?


ReAct 

Reasoning: The LLM explains what it needs to do next.
Example: “To answer this, I must calculate the probability, then look up the definition of variance.”
Acting: The LLM executes a tool call (Python function, API, database query).
Observation: The tool returns results.
Loop: The LLM reasons again, decides the next action, and continues until the final answer is ready.

In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_functions_agent, AgentExecutor, tool
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model="gpt-4", temperature=0)
# Define prompt with both input and agent_scratchpad
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can use tools."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

def search_city(city: str):
    if city.lower() == "france":
        return "Paris"
    return "Unknown"

    
tools = [
    Tool(name="Calculator", func=lambda x: eval(x), description="Do math"),
    Tool(name="Search", func=search_city, description="Search for capitals"),
]

agent = create_openai_functions_agent(llm, tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = agent_executor.invoke({"input": "What is 12*15 and the capital of France?"})
print(result)


result = agent_executor.invoke({"input": "What is 12/15 and the capital of india?"})
print(result)




> Entering new AgentExecutor chain...

Invoking: `Calculator` with `12*15`


180
Invoking: `Search` with `France`


ParisThe result of 12*15 is 180 and the capital of France is Paris.

> Finished chain.
{'input': 'What is 12*15 and the capital of France?', 'output': 'The result of 12*15 is 180 and the capital of France is Paris.'}


> Entering new AgentExecutor chain...

Invoking: `Calculator` with `12/15`


0.8
Invoking: `Search` with `india`


UnknownThe result of 12/15 is 0.8 and the capital of India is New Delhi.

> Finished chain.
{'input': 'What is 12/15 and the capital of india?', 'output': 'The result of 12/15 is 0.8 and the capital of India is New Delhi.'}
